# 16 — Ekspor Model Mandiri untuk Integrasi Aplikasi (Standalone Deployment)

> **Ketentuan Proyek**:
Simpan model terpilih beserta seluruh komponen inferensi agar dapat digunakan oleh aplikasi tanpa menjalankan ulang Jupyter Notebook.

Notebook ini memverifikasi dan mengekspor dua bundle artefak produksi:
1. `models/best_model/` (BGE-M3 Production Engine)
2. `models/candidates/tfidf/` (TF-IDF Lightweight Offline Fallback)

## A. Setup dan Verifikasi Keputusan Pemilihan Model

In [1]:
from pathlib import Path
import sys, json
import pandas as pd

ROOT = Path.cwd().resolve()
if not (ROOT / 'data').exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

decision_path = ROOT / 'outputs' / 'keputusan_model_terpilih.json'
assert decision_path.exists(), 'Jalankan notebook 15 terlebih dahulu'
decision = json.loads(decision_path.read_text(encoding='utf-8'))
print(f'Model Terpilih: {decision["selected_model"]} | Status: {decision["selection_status"]}')

Model Terpilih: bge_m3 | Status: SELECTED_AND_VALIDATED


## B. Pemeriksaan Kelengkapan Artefak Model Terpilih (models/best_model)

In [2]:
best_dir = ROOT / 'models' / 'best_model'
assert best_dir.exists(), f'Direktori {best_dir} tidak ditemukan'

required_best_files = [
    'manifest.json',
    'competency_profiles.csv',
    'catalog.csv',
    'competency_vectors.npy',
    'catalog_vectors.npy',
    'vectorizer.joblib'
]

audit_best = []
for fname in required_best_files:
    fpath = best_dir / fname
    audit_best.append({
        'nama_file': fname,
        'tersedia': fpath.exists(),
        'ukuran_kb': round(fpath.stat().st_size / 1024, 2) if fpath.exists() else 0
    })
df_best_audit = pd.DataFrame(audit_best)
display(df_best_audit)
assert df_best_audit['tersedia'].all(), 'Semua file artefak best_model harus lengkap'

,nama_file,tersedia,ukuran_kb
0,manifest.json,True,0.99
1,competency_profiles.csv,True,40.45
2,catalog.csv,True,1091.31
3,competency_vectors.npy,True,536.12
4,catalog_vectors.npy,True,3892.12
5,vectorizer.joblib,True,1013.09


## C. Pemeriksaan Kelengkapan Artefak Model Fallback (models/candidates/tfidf)

In [3]:
tfidf_dir = ROOT / 'models' / 'candidates' / 'tfidf'
assert tfidf_dir.exists(), f'Direktori {tfidf_dir} tidak ditemukan'

required_tfidf_files = [
    'manifest.json',
    'vectorizer.joblib',
    'catalog.csv',
    'competency_profiles.csv',
    'competency_vectors.npz',
    'catalog_vectors.npz'
]

audit_tfidf = []
for fname in required_tfidf_files:
    fpath = tfidf_dir / fname
    audit_tfidf.append({
        'nama_file': fname,
        'tersedia': fpath.exists(),
        'ukuran_kb': round(fpath.stat().st_size / 1024, 2) if fpath.exists() else 0
    })
df_tfidf_audit = pd.DataFrame(audit_tfidf)
display(df_tfidf_audit)
assert df_tfidf_audit['tersedia'].all(), 'Semua file artefak tfidf harus lengkap'

,nama_file,tersedia,ukuran_kb
0,manifest.json,True,0.20
1,vectorizer.joblib,True,1013.09
2,catalog.csv,True,1091.31
3,competency_profiles.csv,True,40.45
4,competency_vectors.npz,True,8.49
5,catalog_vectors.npz,True,1066.11


## D. Uji Reload Model Tanpa Ketergantungan Notebook

In [4]:
from src.recommender import BookRecommender

# 1. Muat best_model (dengan allow_fallback=True untuk pengujian fleksibel)
model_prod = BookRecommender.load(best_dir, allow_fallback=True)
print(f'Model Prod Dimuat: {model_prod.model_name} (Aktual: {model_prod.actual_model_name}, Fallback: {model_prod.is_fallback_active})')

# 2. Uji inferensi judul + abstrak
test_title = 'Risk Governance and Audit in Healthcare Systems'
test_abstract = 'A detailed methodology for compliance assessment, clinical risk management, and internal audit.'
hasil = model_prod.map_book(test_title, test_abstract, top_k=5)
print('Hasil Inferensi Recommender API:')
display(pd.DataFrame(hasil))
assert len(hasil) >= 1
print('PASS — Artefak model mandiri berhasil diekspor dan diverifikasi.')


Model Prod Dimuat: bge_m3 (Aktual: bge_m3, Fallback: False)


c:\Users\AUREL ELVIOLITA\Downloads\AI_eLibrary\E-book-Library--main\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 391/391 [00:00<00:00, 8655.72it/s]


Hasil Inferensi Recommender API:


,rank,competency_id,competency_name,score,confidence,model,model_type,is_fallback,latency_breakdown_ms
0,1,C024,Internal Compliance,0.6680,SANGAT_TINGGI,bge_m3,dense_biencoder,False,"{'text_cleaning_ms': 0.13, 'embedding_encoding..."
1,2,C027,Regulation Management,0.6116,TINGGI,bge_m3,dense_biencoder,False,"{'text_cleaning_ms': 0.13, 'embedding_encoding..."
2,3,C025,Auditing Skill,0.5991,TINGGI,bge_m3,dense_biencoder,False,"{'text_cleaning_ms': 0.13, 'embedding_encoding..."
3,4,C008,Utilization Management,0.5971,TINGGI,bge_m3,dense_biencoder,False,"{'text_cleaning_ms': 0.13, 'embedding_encoding..."
4,5,C015,Quality Management,0.5948,TINGGI,bge_m3,dense_biencoder,False,"{'text_cleaning_ms': 0.13, 'embedding_encoding..."


PASS — Artefak model mandiri berhasil diekspor dan diverifikasi.


## Kesimpulan Tahap

Seluruh komponen inferensi mandiri telah tersimpan di direktori `models/best_model/` dan `models/candidates/tfidf/`, siap dipakai oleh backend aplikasi tanpa notebook.